# F1 Strategy Predictor
## Real-Time Race Simulator

---

Simulates F1 races using trained LSTM models to predict:
1. **Pit Stop Timing**: Probability of pit for each lap
2. **Tire Compound**: SOFT, MEDIUM, HARD, INTERMEDIATE

### Requirements
- `f1_pit_model.keras`, `f1_compound_model.keras`,  `f1_pit_scaler.pkl`, `f1_comp_scaler.pkl`, `label_encoder.pkl`, `modelConfig.json`, `f1_dataset_featured.pkl`

# 1. Setup

In [ ]:
import importlib.util
if importlib.util.find_spec('fastf1') is None:
    !pip install fastf1 --quiet

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os

import numpy as np

import pandas as pd

from io import BytesIO
import requests
import os
import tempfile

import matplotlib.pyplot as plt

import json

import joblib

from tensorflow.keras.models import load_model

In [ ]:
# Compound colors and labels
COMPOUND_COLORS = {
    'SOFT': '#FF3333',
    'MEDIUM': '#FFD700',
    'HARD': '#E8E8E8',
    'INTERMEDIATE': '#39FF14'
}
COMPOUND_SHORT = {'SOFT': 'S', 'MEDIUM': 'M', 'HARD': 'H', 'INTERMEDIATE': 'I'}

In [ ]:
BASE_URL = (
    "https://raw.githubusercontent.com/"
    "FedericoSabbadini/f1-strategy-predictor/"
    "main/03:model/output/Model/"
)

# Sends an HTTP GET request
def download_bytes(url: str) -> bytes:
    r = requests.get(url)
    r.raise_for_status()
    return r.content
# TF load_model using download_bytes
def load_keras_from_url(url: str):
    with tempfile.NamedTemporaryFile(suffix=".keras", delete=False) as f:
        f.write(download_bytes(url))
        path = f.name
    model = load_model(path)
    os.remove(path)
    return model

In [ ]:
model_pit = load_keras_from_url(BASE_URL + "f1_pit_model.keras")
model_comp = load_keras_from_url(BASE_URL + "f1_compound_model.keras")

scaler_pit = joblib.load(BytesIO(download_bytes(BASE_URL + "f1_pit_scaler.pkl")))
scaler_comp = joblib.load(BytesIO(download_bytes(BASE_URL + "f1_comp_scaler.pkl")))

label_encoder = joblib.load(BytesIO(download_bytes(BASE_URL + "label_encoder.pkl")))

df_f1 = pd.read_pickle(BytesIO(download_bytes(BASE_URL + "f1_dataset_featured.pkl")))

config = json.loads(download_bytes(BASE_URL + "modelConfig.json"))


In [ ]:
# Config
SEQ_LEN = config['sequence_length']
FEATURES = config['FEATURES']
PIT_THRESHOLD_ADD = 0.21
PIT_THRESHOLD = config['pit_threshold'] + PIT_THRESHOLD_ADD # best simulation accuracy

---
# 2. Race Selection

Run cells below to see available options.

### 2.1 Available Years

In [ ]:
years = sorted(df_f1['Year'].unique())
print('Available years:')
print('-' * 40)
for year in years:
    print(f'  {year}')

*Select year:* [ex. YEAR = 2023]

In [ ]:
YEAR =

### 2.2 Available Races

In [ ]:
races = df_f1[df_f1['Year'] == YEAR][['Round', 'RaceName']].drop_duplicates().sort_values('Round')
print(f'Races {YEAR}:')
print('-' * 40)
for _, row in races.iterrows():
    print(f"  {int(row['Round']):>2}. {row['RaceName']}")

*Select race:* [ex. ROUND = 21]

In [ ]:
ROUND =

### 2.3 Available Drivers

In [ ]:
race_df = df_f1[(df_f1['Year'] == YEAR) & (df_f1['Round'] == ROUND)]
race_name = race_df['RaceName'].iloc[0] if len(race_df) > 0 else 'N/A'

drivers = race_df[['Driver', 'Team']].drop_duplicates().sort_values('Team')
print(f'{race_name} {YEAR}')
print('-' * 40)
print(f'{"Driver":<8} {"Team":<25}')
print('-' * 40)
for _, row in drivers.iterrows():
    print(f"{row['Driver']:<8} {row['Team']:<25}")

*Select driver:* [ex. DRIVER = 'LAW']

In [ ]:
DRIVER = ''

### 2.4 Real Race Summary

In [ ]:
race_data = df_f1[(df_f1['Year'] == YEAR) & (df_f1['Round'] == ROUND)].copy()
race_name = race_data['RaceName'].iloc[0] if len(race_data) > 0 else None

if race_name and DRIVER in race_data['Driver'].values:
    driver_data = race_data[race_data['Driver'] == DRIVER].sort_values('LapNumber')
    total_laps = int(driver_data['LapNumber'].max())
    team = driver_data['Team'].iloc[0]

    # Build strategy string
    stints_info = []
    n_prev = 0
    stints_info.append(f"start(0)-- ")
    stint_number = len(sorted(driver_data['Stint'].unique()))
    for stint in sorted(driver_data['Stint'].unique()):
        stint_df = driver_data[driver_data['Stint'] == stint]
        compound = stint_df['Compound'].iloc[0]
        n_laps = int(stint_df['LapNumber'].max())
        string_lap = f"{COMPOUND_SHORT.get(compound, '?')}x{n_laps-n_prev} "
        if stint < int(stint_number):
          string_pit = f"--PIT({n_laps+1})--> "
        else:
          string_pit = f"--end({n_laps})"
        string = string_lap + string_pit
        stints_info.append(string)
        n_prev=n_laps

    # Safety car laps
    sc_laps = []
    if 'UnderCaution' in driver_data.columns:
        sc_laps = driver_data[driver_data['UnderCaution'] == 1]['LapNumber'].astype(int).tolist()

    print('=' * 50)
    print(f'  {race_name} {YEAR}')
    print(f'  {DRIVER} - {team}')
    print('=' * 50)
    print(f'  Total laps: {total_laps}')
    print(f'  Stints: {driver_data["Stint"].nunique()}')
    strategy_str = "".join(stints_info)
    print(f'  Strategy: {strategy_str}')
    if sc_laps:
        print(f'  Under Safety Car: {len(sc_laps)} laps')
    print('=' * 50)
else:
    print('ERROR: Race or driver not found')
    print(f'  Year: {YEAR}, Round: {ROUND}, Driver: {DRIVER}')

---
# 3. Simulation

Processes each lap, builds sequences, and generates predictions.

In [ ]:
class RaceSimulator:
    """
    Full lap-by-lap race simulator.
    """


    def __init__(
        self,
        model_pit,
        model_comp,
        scaler_pit,
        scaler_comp,
        label_encoder,
        features,
        seq_len,
        pit_threshold,
        pit_thresholdAdd,
    ):
        # Store models and preprocessing tools
        self.model_pit = model_pit
        self.model_comp = model_comp
        self.scaler_pit = scaler_pit
        self.scaler_comp = scaler_comp
        self.label_encoder = label_encoder

        # Model configuration
        self.features = features
        self.seq_len = seq_len
        self.pit_threshold = pit_threshold
        self.pit_thresholdAdd = pit_thresholdAdd


    def _make_sequence(self, history_df, scaler):
        """
        Turn past laps into an LSTM input tensor.
        Shape: (1, seq_len, n_features)
        """

        # Giro 0, no past data = fully zero sequence (remember padding/masking)
        if len(history_df) == 0:
            return np.zeros((1, self.seq_len, len(self.features)), dtype=np.float32)

        # Extract and scale feature values
        X = history_df[self.features].values.astype(np.float32)
        X = scaler.transform(X)

        # Left-pad with 0 if too short
        if len(X) < self.seq_len:
            pad = np.zeros((self.seq_len - len(X), X.shape[1]), dtype=np.float32)
            X = np.vstack([pad, X])
        else:
            # Keep only last seq_len's laps
            X = X[-self.seq_len:]

        # Add batch dimension
        return X.reshape(1, self.seq_len, X.shape[1])


    def simulate(self, race_df, driver):
        """
        Simulate the race lap by lap for one driver.
        """

        # Filter and sort driver laps
        df = (
            race_df[race_df["Driver"] == driver]
            .sort_values("LapNumber")
            .reset_index(drop=True)
        )

        total_laps = int(df["LapNumber"].max())
        num_stints = int(df["Stint"].max())

        # Start of a new stint (lap)
        pit_laps = set()
        pit_laps_window = set()
        for s in range(2, num_stints + 1):
            first_lap = int(df[df["Stint"] == s - 1]["LapNumber"].max())
            pit_laps.add(first_lap)
            pit_laps_window.add(first_lap-1)


        # Safety-car laps
        sc_laps = set(
            df[df.get("UnderCaution", 0) == 1]["LapNumber"].astype(int)
        )

        results = []

        # Process lap by lap
        for i in range(len(df)):
            row = df.iloc[i]
            lap = int(row["LapNumber"])
            stint = int(row["Stint"])

            # History = all previous laps in this stint
            history = df[(df["Stint"] == stint) & (df["LapNumber"] < lap)]

            # --------------------
            # PIT PROBABILITY
            # --------------------
            seq = self._make_sequence(history, self.scaler_pit)
            # Predict se Pit in 3 Laps or Not
            pit_prob = float(self.model_pit.predict(seq, verbose=0).ravel()[0])
            pit_pred = int(pit_prob >= self.pit_threshold) # Threshold applied

            # --------------------
            # COMPOUND PREDICTION
            # --------------------
            comp_pred = None
            comp_actual = None

            # Only predict compound if this lap is a pit lap, to make easy the comparison
            # Predict compound
            prev_stint_df = df[df["LapNumber"] < lap]
            seq = self._make_sequence(prev_stint_df, self.scaler_comp)

            probs = self.model_comp.predict(seq, verbose=0)[0]
            comp_pred = self.label_encoder.inverse_transform(
                [probs.argmax()]
            )[0]

            comp_actual = row["Compound"]

            # --------------------
            # STORE RESULT
            # --------------------
            results.append({
                "Lap": lap,
                "TotalLaps": total_laps,
                "Stint": stint,
                "Compound": row["Compound"],
                "TyreAge": int(row.get("TyreLife", 0)),
                "PitProb": pit_prob,
                "PitPred": pit_pred,
                "PitThisLap": int(lap in pit_laps_window),
                "IsLastStint": stint == num_stints,
                "CompPred": comp_pred,
                "CompActual": comp_actual,
                "UnderCaution": int(row.get("UnderCaution", 0)),
                "DataAvailable": True,
                "MissingReason": None
            })

        result_df = pd.DataFrame(results)

        # --------------------
        # RECOMMENDED PIT LAPS
        # --------------------
        recommended = {}
        THRS = self.pit_threshold
        for s in range(1, num_stints):
            rows = result_df[
                (result_df["Stint"] == s)
            ]
            rows_probMax = rows[rows['PitProb']>THRS].copy()
            rows_probNotMax = rows[rows['PitProb']>THRS-(self.pit_thresholdAdd/2)].copy()
            if len(rows_probMax) > 0:
                idx = rows_probMax["PitProb"].idxmin()
                recommended[s] = {
                    "lap": int(rows_probMax.loc[idx, "Lap"]),
                    "prob": float(rows_probMax.loc[idx, "PitProb"])
                }
            elif len(rows_probNotMax) > 0:
                THRS = THRS - (self.pit_thresholdAdd/2)
                idx = rows_probNotMax["PitProb"].idxmin()
                recommended[s] = {
                    "lap": int(rows_probNotMax.loc[idx, "Lap"]),
                    "prob": float(rows_probNotMax.loc[idx, "PitProb"])
                }
            else:
               prob = rows['PitProb'].max()
               rows_prob = rows[rows['PitProb']==prob].copy()
               THRS = THRS - self.pit_thresholdAdd
               if len(rows_prob) > 0 and prob>=self.pit_threshold-self.pit_thresholdAdd:
                idx = rows_prob["PitProb"].idxmin()
                recommended[s] = {
                    "lap": int(rows_prob.loc[idx, "Lap"]),
                    "prob": float(rows_prob.loc[idx, "PitProb"])
                }


        return result_df, pit_laps, sc_laps, recommended, num_stints, THRS


# Run simulation
simulator = RaceSimulator(
    model_pit, model_comp, scaler_pit, scaler_comp, label_encoder,
    FEATURES, SEQ_LEN, PIT_THRESHOLD, PIT_THRESHOLD_ADD
)

sim_df, pit_laps, sc_laps, recommended_pits, num_stints, THRS = simulator.simulate(race_data, DRIVER)


---
# 4. Results

Compare predicted vs actual pit stops.

In [ ]:
print('\n' + '=' * 60)
print('         RECOMMENDED vs ACTUAL PIT STOPS')
print('=' * 60)
max_diff = 0

for stint in range(1, num_stints + 1):
    stint_df = sim_df[sim_df['Stint'] == stint]
    stint_compound = stint_df['Compound'].iloc[0]

    next_compound_rows = sim_df[sim_df['Stint'] == stint + 1]
    if not next_compound_rows.empty:
        comp_real = next_compound_rows['Compound'].iloc[0]
    else:
        comp_real = None

    print(f'\nStint {stint}')
    print('-' * 60)

    if stint == num_stints:
        print('  Final stint - end of race')
        continue

    if stint not in recommended_pits:
        print('  Insufficient data')
        continue

    # Recommended pit lap
    rec = recommended_pits[stint]

    # Actual pit info
    actual_rows = sim_df[(sim_df['Stint'] == stint) & (sim_df['PitThisLap'] == 1)]

    # There is an actual pit
    actual_lap = int(actual_rows['Lap'].iloc[0]) + 2
    diff = actual_lap - rec['lap']
    max_diff = max(max_diff, abs(diff))

    comp_pred = actual_rows['CompPred'].iloc[0]
    # Matches
    match_pit = '✅' if abs(diff) < 4 else '❌'
    match_comp = '✅' if comp_real is not None and comp_pred == comp_real else '❌'

    print(f'  {"Real Pit:":<26} Lap {actual_lap}')
    print(f'  {"Predicted Pit:":<26} Lap {rec["lap"]} (P = {rec["prob"]:.0%})  {match_pit} (Δ = {diff:+d})')
    print(f'\n  {"Real Next Compound:":<26} Tyre {comp_real}')
    print(f'  {"Predicted Next Compound:":<26} Tyre {comp_pred}  {match_comp}')

print('\n' + '=' * 60)

---
# 5. Visualization

Plot pit probability and strategy across the race.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), height_ratios=[3, 1])
data = sim_df[sim_df['DataAvailable']]
total_laps = int(sim_df['TotalLaps'].iloc[0])
STAR_SIZE = 200
pit_laps_sorted = sorted(list(pit_laps))

recommended_pits_sorted = sorted(list(recommended_pits.values()), key=lambda x: x['lap'])
recommended_laps = [rec['lap'] for rec in recommended_pits_sorted]

# ====== TOP PLOT: Pit Probability ======
ax1 = axes[0]

# Extend P(Pit) curve to start from lap 0
laps_extended = [0] + data['Lap'].tolist()
probs_extended = [0] + data['PitProb'].tolist()

# P(Pit) curve
ax1.plot(laps_extended, probs_extended, 'b-', linewidth=2, label='P(Pit)')
ax1.fill_between(laps_extended, 0, probs_extended, alpha=0.2, color='blue')

# Threshold line
ax1.axhline(y=THRS, color='red', linestyle='--',
            linewidth=1.5, alpha=0.7, label=f'Threshold ({THRS:.0%})')

# Actual pit stops (stars)
for i, pit in enumerate(pit_laps_sorted):
    # Interpolate probability at pit lap
    pit = pit + 1
    if pit < data['Lap'].min() or pit > data['Lap'].max():
        prob = THRS
    else:
        prob = np.interp(pit, data['Lap'].values, data['PitProb'].values)

    ax1.scatter([pit], [prob], color='gold', s=STAR_SIZE, marker='*',
                zorder=6, edgecolors='black', linewidths=1,
                label='Real Pit' if i == 0 else '')

# Predicted pit stops (green lines)
for i, info in enumerate(recommended_pits.values()):
    ax1.axvline(x=info['lap'], color='green', linewidth=2, alpha=0.7,
                label='Predicted Pit' if i == 0 else '')

ax1.set_xlim(0, total_laps)
ax1.set_ylim(0, 1.1)
ax1.set_ylabel('P(Pit)')
ax1.set_title(f'{DRIVER} - {race_name} {YEAR}')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# ====== BOTTOM PLOT: Tire Strategy ======
ax2 = axes[1]
sim_df_sorted = sim_df.sort_values(['Stint', 'Lap'])

# Actual strategy (solid lines)
for stint_int, sdata in sim_df_sorted.groupby('Stint'):
    compound = sdata['Compound'].iloc[0]
    color = COMPOUND_COLORS.get(compound, 'gray')
    min_lap = max(laps)
    laps = sdata['Lap'].tolist()

    # Original tyre age for actual data
    tyre_age = list(range(len(laps)))
    # For first stint, fill missing laps from 0 to min_lap - 1
    if stint_int == 1:
        min_lap = laps[0]
        max_lap = laps[-1]

        # Extend backwards to lap 0 if needed
        if min_lap > 0:
            laps = list(range(0, min_lap)) + laps

        # Include last lap + 1
        laps = laps + [max_lap + 1]

        # Normalize to fill any missing laps
        lap_start = laps[0]
        lap_end = laps[-1]
        laps = list(range(lap_start, lap_end + 1))

        # Continuous tyre age
        tyre_age = [lap - lap_start for lap in laps]

    else:
        # Other stints, add extra laps before stint for plotting continuity
        max_lap = max(laps) + 1
        for i in range(min_lap, max_lap + 1):
            if i not in laps:
                laps.append(i)
        laps = sorted(laps)
        tyre_age = [lap - laps[0] for lap in laps]

    ax2.plot(laps, tyre_age, color=color, linewidth=4, zorder=2)


# Build predicted compounds dict
pred_compounds = {}
for pit_lap in recommended_laps:
    pit_row = sim_df[sim_df['Lap'] == pit_lap]
    next_stint = int(pit_row['Stint'].iloc[0]) + 1
    pred_compound = pit_row['CompPred'].iloc[0]
    pred_compounds[next_stint] = pred_compound

# Predicted strategy (dashed lines)
pred_pits_sorted = []
for stint, info in recommended_pits.items():
    next_stint = stint + 1
    if next_stint in pred_compounds:
        pred_pits_sorted.append((info['lap'], pred_compounds[next_stint]))

pred_pits_sorted = sorted(pred_pits_sorted, key=lambda x: x[0])
for i, (pred_lap, pred_compound) in enumerate(pred_pits_sorted):
    next_lap = pred_pits_sorted[i + 1][0] if i + 1 < len(pred_pits_sorted) else total_laps
    pred_laps = list(range(pred_lap, next_lap + 1))
    pred_tyre_age = list(range(len(pred_laps)))
    color = COMPOUND_COLORS.get(pred_compound, 'gray')
    ax2.plot(pred_laps, pred_tyre_age, color=color, linewidth=2,
             linestyle='--', zorder=3)
# Pit markers
STAR_Y = 1
for pit in pit_laps_sorted:
      pit = pit + 1
      ax2.scatter(pit, STAR_Y, color='gold', s=STAR_SIZE, marker='*',
                zorder=6, edgecolors='black', linewidths=1)

for info in recommended_pits.values():
    ax2.axvline(x=info['lap'], color='green', linestyle='--',
                linewidth=1.5, alpha=0.5)

# Legend
ax2.plot([], [], color='gray', linewidth=4, label='Real')
ax2.plot([], [], color='gray', linewidth=2, linestyle='--', label='Predicted')

ax2.set_xlim(0, total_laps)
ax2.set_ylim(0, None)
ax2.set_xlabel('Lap')
ax2.set_ylabel('Tire Age')
ax2.legend(loc='upper left', fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


---
# 6. Save Results

Export simulation to CSV and chart to PNG.

In [ ]:
dirname = f"{YEAR}_R{ROUND}_{DRIVER}"
os.makedirs(dirname, exist_ok=True)

In [ ]:
# Full simulation CSV
sim_df.to_csv(os.path.join(dirname, 'simulation.csv'), index=False)

# Summary CSV
pit_summary = []
for stint in range(1, num_stints + 1):
    row = {
        'Stint': stint,
        'Compound': sim_df[sim_df['Stint'] == stint]['Compound'].iloc[0]
    }

    if stint == num_stints:
        row.update({
            'RecLap': None, 'RecProb': None, 'ActualLap': None,
            'Diff': None, 'NextCompound': None, 'PredCompound': None,
            'CompMatch': None
        })
    elif stint in recommended_pits:
        rec = recommended_pits[stint]
        row['RecLap'] = rec['lap']
        row['RecProb'] = round(rec['prob'], 3)

        pit_rows = sim_df[
            (sim_df['Stint'] == stint + 1) &
            (sim_df['PitThisLap'] == 1)
        ]

        if len(pit_rows) > 0:
            actual = int(pit_rows['Lap'].iloc[0])
            row['ActualLap'] = actual
            row['Diff'] = actual - rec['lap']
            row['NextCompound'] = pit_rows['CompActual'].iloc[0]
            row['PredCompound'] = pit_rows['CompPred'].iloc[0]
            row['CompMatch'] = row['NextCompound'] == row['PredCompound']

    pit_summary.append(row)

pd.DataFrame(pit_summary).to_csv(os.path.join(dirname, 'summary.csv'), index=False)

# Save chart
fig.savefig(os.path.join(dirname, 'chart.png'), dpi=150, bbox_inches='tight')

print('\nFiles saved:')
print(f'  ✓ {dirname}/simulation.csv')
print(f'  ✓ {dirname}/summary.csv')
print(f'  ✓ {dirname}/chart.png')